# Lección 04: Casos Reales — Pipeline completo de clasificación

Aplicamos todo lo aprendido en el módulo para construir un pipeline completo: cargamos el dataset Breast Cancer Wisconsin, entrenamos dos modelos (KNN y LogisticRegression), comparamos métricas y elegimos el más adecuado para un contexto médico.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=2, suppress=True)

## Cargar el dataset Breast Cancer Wisconsin

Este dataset contiene 569 casos de biopsias con 30 features numéricas. Cada caso está etiquetado como **maligno** o **benigno**. Separamos en entrenamiento (80%) y prueba (20%) usando estratificación para mantener la proporción de clases.

In [ ]:
data = load_breast_cancer()
X = data.data
y = data.target
feature_names = data.feature_names
target_names = data.target_names

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Muestras totales: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Clases: {target_names}")
print(f"Train: {X_train.shape[0]} muestras")
print(f"Test: {X_test.shape[0]} muestras")

## Escalar los datos

KNN depende de distancias, así que escalamos las features para que ninguna domine por su magnitud. LogisticRegression también se beneficia del escalado para converger más rápido.

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"Media de la primera feature (train): {X_train_s[:,0].mean():.3f}")
print(f"Std de la primera feature (train): {X_train_s[:,0].std():.3f}")

## Entrenar un clasificador KNN (k=5)

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_s, y_train)
y_pred_knn = knn.predict(X_test_s)

print(f"KNN accuracy: {knn.score(X_test_s, y_test):.3f}")

## Matriz de confusión y métricas para KNN

Calculamos accuracy, precision, recall y F1-score. En un contexto médico, minimizar los falsos negativos (pacientes con cáncer clasificados como benignos) suele ser crítico.

In [ ]:
cm_knn = confusion_matrix(y_test, y_pred_knn)
print("Confusion matrix (KNN):")
print(cm_knn)

acc_knn = accuracy_score(y_test, y_pred_knn)
pre_knn = precision_score(y_test, y_pred_knn)
rec_knn = recall_score(y_test, y_pred_knn)
f1_knn = f1_score(y_test, y_pred_knn)

print(f"\nAccuracy:  {acc_knn:.3f}")
print(f"Precision: {pre_knn:.3f}")
print(f"Recall:    {rec_knn:.3f}")
print(f"F1-score:  {f1_knn:.3f}")

print(f"\nInterpretación: {cm_knn[0,1]} falsos negativos y {cm_knn[1,0]} falsos positivos.")

## Entrenar LogisticRegression en el mismo dataset

In [ ]:
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_s, y_train)
y_pred_lr = logreg.predict(X_test_s)

cm_lr = confusion_matrix(y_test, y_pred_lr)
print(f"Logistic Regression accuracy: {logreg.score(X_test_s, y_test):.3f}\n")
print("Confusion matrix (Logistic Regression):")
print(cm_lr)

acc_lr = accuracy_score(y_test, y_pred_lr)
pre_lr = precision_score(y_test, y_pred_lr)
rec_lr = recall_score(y_test, y_pred_lr)
f1_lr = f1_score(y_test, y_pred_lr)

print(f"\nAccuracy:  {acc_lr:.3f}")
print(f"Precision: {pre_lr:.3f}")
print(f"Recall:    {rec_lr:.3f}")
print(f"F1-score:  {f1_lr:.3f}")

## Comparación de modelos: tabla de métricas

In [ ]:
metrics = {
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score"],
    "KNN": [acc_knn, pre_knn, rec_knn, f1_knn],
    "LogisticRegression": [acc_lr, pre_lr, rec_lr, f1_lr],
}

df_metrics = pd.DataFrame(metrics).round(3)
print(df_metrics)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(df_metrics))
width = 0.35

ax.bar(x - width/2, df_metrics["KNN"], width, label="KNN", color="#3498db")
ax.bar(x + width/2, df_metrics["LogisticRegression"], width, label="LogisticRegression", color="#2ecc71")

ax.set_ylabel("Score")
ax.set_title("Comparación de métricas: KNN vs LogisticRegression")
ax.set_xticks(x)
ax.set_xticklabels(df_metrics["Metric"], rotation=15)
ax.legend()
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig("metrics_comparison.png", dpi=100, bbox_inches="tight")
plt.show()

## Resumen

- Construimos un pipeline completo: carga → split estratificado → escalado → entrenamiento → evaluación → comparación.
- Entrenamos **KNN (k=5)** y **LogisticRegression** sobre el mismo dataset.
- Reportamos accuracy, precision, recall y F1-score para ambos modelos.
- En un problema médico, **recall** es especialmente importante porque los falsos negativos pueden retrasar un diagnóstico de cáncer.
- La elección final depende del costo relativo de los errores: minimizar falsos negativos vs. evitar falsos positivos que generen biopsias innecesarias.